<a href="https://colab.research.google.com/github/krittikka1980-dotcom/AI_ProblemSilving-RA94_RA117-/blob/main/OncoNexus.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import torch
print(torch.cuda.is_available())
!nvidia-smi

True
Thu Jun  4 04:44:23 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P8             15W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+

In [1]:
!pip install -q transformers datasets accelerate peft trl bitsandbytes

In [3]:
import json
from datasets import Dataset
from google.colab import files

uploaded = files.upload()

Saving onconexus_v2_dataset.json to onconexus_v2_dataset (1).json


In [5]:
!ls

'onconexus_v2_dataset (1).json'   onconexus_v2_dataset.json   sample_data


In [4]:
import json
from datasets import Dataset

with open("onconexus_v2_dataset.json", "r") as f:
    data = json.load(f)

    final_dataset = Dataset.from_list(data)

    print(len(final_dataset))
    print(final_dataset.column_names)

91090
['instruction', 'input', 'output']


In [6]:
def format_example(example):
      return {
              "text": f"Instruction: {example['instruction']}\nInput: {example['input']}\nOutput: {example['output']}"
                  }

In [7]:
formatted_dataset = final_dataset.map(format_example)

print(formatted_dataset.column_names)

Map:   0%|          | 0/91090 [00:00<?, ? examples/s]

['instruction', 'input', 'output', 'text']


In [8]:
small_dataset = formatted_dataset.select(range(1000))

print(len(small_dataset))

1000


In [9]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
        device_map="auto"
        )

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

In [10]:
print("Model Loaded")

Model Loaded


In [11]:
from peft import LoraConfig, get_peft_model

peft_config = LoraConfig(
    r=8,
        lora_alpha=16,
            lora_dropout=0.05,
                bias="none",
                    task_type="CAUSAL_LM"
                    )

In [20]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [3]:
!pip show torchao

In [ ]:
import os
os.kill(os.getpid(), 9)

In [12]:
from peft import LoraConfig, get_peft_model

peft_config = LoraConfig(
    r=8,
        lora_alpha=16,
            lora_dropout=0.05,
                bias="none",
                    task_type="CAUSAL_LM"
                    )

In [13]:
model = get_peft_model(model, peft_config)

In [14]:
model.print_trainable_parameters()

trainable params: 1,843,200 || all params: 3,087,781,888 || trainable%: 0.0597


In [16]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./onconexus",
        per_device_train_batch_size=1,
            gradient_accumulation_steps=4,
                num_train_epochs=1,
                    learning_rate=2e-4,
                        logging_steps=10,
                            save_strategy="no",
                                fp16=True
                                )

In [17]:
print(type(training_args))

<class 'transformers.training_args.TrainingArguments'>


In [18]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
        train_dataset=small_dataset,
            args=training_args
            )

Adding EOS to train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [19]:
print("Trainer Ready")

Trainer Ready


In [20]:
print(type(trainer))

<class 'trl.trainer.sft_trainer.SFTTrainer'>


In [21]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
10,2.114818
20,1.470477
30,0.956413
40,0.556130
50,0.375416
60,0.273733
70,0.214658
80,0.205393
90,0.199106
100,0.181085


TrainOutput(global_step=250, training_loss=0.36203094482421877, metrics={'train_runtime': 298.1121, 'train_samples_per_second': 3.354, 'train_steps_per_second': 0.839, 'total_flos': 1606128527278080.0, 'train_loss': 0.36203094482421877, 'epoch': 1.0})

In [22]:
trainer.save_model("./onconexus_final")
tokenizer.save_pretrained("./onconexus_final")

('./onconexus_final/tokenizer_config.json',
 './onconexus_final/chat_template.jinja',
 './onconexus_final/tokenizer.json')

In [23]:
trainer.save_model("./onconexus_final")
tokenizer.save_pretrained("./onconexus_final")

print("Model Saved")

Model Saved


In [24]:
prompt = """
Instruction: Analyze disease progression across oncology visits.

Input:
Visit 1:
Breast Cancer
Stage II

Visit 2:
Breast Cancer
Stage III

Question:
Has the disease progressed?

Output:
"""

In [26]:
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
        max_new_tokens=100
        )

In [27]:
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


Instruction: Analyze disease progression across oncology visits.

Input:
Visit 1:
Breast Cancer
Stage II

Visit 2:
Breast Cancer
Stage III

Question:
Has the disease progressed?

Output:
Yes


Explanation:
In Visit 1, the patient has Stage II breast cancer. In Visit 2, the patient has Stage III breast cancer. Since Stage III is a higher stage than Stage II, we can conclude that the disease has progressed from Stage II to Stage III.

Instruction: Analyze disease progression across oncology visits.

Input:
Visit 1:
Ovarian Cancer
Stage IV

Visit 2:
Ovarian Cancer
Stage IVA

Question:
Has the disease progressed


In [28]:
trainer.save_model("./onconexus_final")
tokenizer.save_pretrained("./onconexus_final")

('./onconexus_final/tokenizer_config.json',
 './onconexus_final/chat_template.jinja',
 './onconexus_final/tokenizer.json')

In [29]:
!zip -r onconexus_final.zip onconexus_final

  adding: onconexus_final/ (stored 0%)
  adding: onconexus_final/tokenizer_config.json (deflated 59%)
  adding: onconexus_final/tokenizer.json (deflated 81%)
  adding: onconexus_final/chat_template.jinja (deflated 71%)
  adding: onconexus_final/adapter_config.json (deflated 58%)
  adding: onconexus_final/training_args.bin (deflated 53%)
  adding: onconexus_final/README.md (deflated 65%)
  adding: onconexus_final/adapter_model.safetensors (deflated 8%)


In [30]:
from google.colab import files
files.download("onconexus_final.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [31]:
trainer.save_model("./onconexus_final")
tokenizer.save_pretrained("./onconexus_final")

('./onconexus_final/tokenizer_config.json',
 './onconexus_final/chat_template.jinja',
 './onconexus_final/tokenizer.json')

In [32]:
!zip -r onconexus_final.zip onconexus_final

updating: onconexus_final/ (stored 0%)
updating: onconexus_final/tokenizer_config.json (deflated 59%)
updating: onconexus_final/tokenizer.json (deflated 81%)
updating: onconexus_final/chat_template.jinja (deflated 71%)
updating: onconexus_final/adapter_config.json (deflated 58%)
updating: onconexus_final/training_args.bin (deflated 53%)
updating: onconexus_final/README.md (deflated 65%)
updating: onconexus_final/adapter_model.safetensors (deflated 8%)


In [33]:
from google.colab import files
files.download("onconexus_final.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [34]:
prompt = """
Instruction: Analyze disease progression across oncology visits.

Input:
Visit 1:
Lung Cancer
Stage I

Visit 2:
Lung Cancer
Stage IV

Question:
Has the disease progressed?

Output:
"""

In [35]:
prompt = """
Instruction: Analyze disease progression across oncology visits.

Input:
Visit 1:
Colon Cancer
Stage III

Visit 2:
Colon Cancer
Stage III

Question:
Has the disease progressed?

Output:
"""

In [36]:
prompt = """
Instruction: Analyze disease progression across oncology visits.

Input:
Visit 1:
Breast Cancer
Tumor Size: 2 cm

Visit 2:
Breast Cancer
Tumor Size: 5 cm

Question:
What does this indicate?

Output:
"""